<a href="https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

In [ ]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [ ]:
%pip -q install -U duckdb huggingface_hub

import duckdb
from google.colab import userdata
from huggingface_hub import whoami

token = userdata.get("HF_TOKEN")

print("Token found:", token is not None)
print("Logged in as:", whoami(token=token)["name"])

Token found: True
Logged in as: srijan317


In [ ]:
import pandas as pd
dataset = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_avg_position,
        gsc_impressions,
        gsc_clicks,
        ga4_pageviews,
        ga4_sessions,
        month
    FROM {REL}
""").df()

# Calculate CTR cleanly in Pandas to avoid division by zero issues
dataset['ctr'] = dataset['gsc_clicks'] / dataset['gsc_impressions'].replace(0, pd.NA)
dataset['ctr'] = dataset['ctr'].fillna(0)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_1038/39244094.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset['ctr'] = dataset['ctr'].fillna(0)


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

The dataset displays extreme zero-inflation and heavy right-skewness. Over 75% of daily records have 0 clicks and 0 GA4 pageviews, with a mean impression count of 28.5 vs. a median of 0. Furthermore, gsc_avg_position contains ~63% missing values corresponding to days with zero search impressions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import statistics
import numpy as np
import pandas as pd

mean_impressions = statistics.mean(dataset['gsc_impressions'])
median_impressions = statistics.median(dataset['gsc_impressions'])

mean_clicks = statistics.mean(dataset['ctr'])
median_clicks = statistics.median(dataset['ctr'])

print("Mean Impressions:", mean_impressions)
print("Median Impressions:", median_impressions)
print("Mean CTR:", mean_clicks)
print("Median CTR:", median_clicks)

Mean Impressions: 28.51811900731788
Median Impressions: 0.0
Mean CTR: 0.0011304078142477117
Median CTR: 0.0


In [ ]:
ctr_min = np.min(dataset['ctr'])
ctr_max = np.max(dataset['ctr'])
q1, q2, q3 = np.percentile(dataset['ctr'], [25, 50, 75])

position_min = np.min(dataset['gsc_avg_position'])
position_max = np.max(dataset['gsc_avg_position'])
q4, q5, q6 = np.percentile(dataset['gsc_avg_position'], [25, 50, 75])

print("CTR Min:", ctr_min)
print("CTR Max:", ctr_max)
print("CTR Q1:", q1)
print("CTR Q2:", q2)
print("CTR Q3:", q3)

print("Position Min:", position_min)
print("Position Max:", position_max)
print("Position Q1:", q4)
print("Position Q2:", q5)
print("Position Q3:", q6)

CTR Min: 0.0
CTR Max: 1.0
CTR Q1: 0.0
CTR Q2: 0.0
CTR Q3: 0.0
Position Min: 0.0
Position Max: 498.0
Position Q1: nan
Position Q2: nan
Position Q3: nan


In [ ]:
dataset.describe()

,report_date,gsc_avg_position,gsc_impressions,gsc_clicks,ga4_pageviews,ga4_sessions,ctr
count,9841378,3.611061e+06,9.841378e+06,9.841378e+06,6822637.0,6822637.0,9.841378e+06
mean,2026-03-16 06:36:22.440222,1.582665e+01,2.851812e+01,8.350782e-02,0.217637,0.190514,1.130408e-03
min,2026-03-01 00:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00
25%,2026-03-09 00:00:00,3.742120e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00
50%,2026-03-16 00:00:00,7.500000e+00,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00
75%,2026-03-24 00:00:00,2.020000e+01,6.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00
max,2026-03-31 00:00:00,4.980000e+02,4.008400e+04,2.740000e+02,875.0,792.0,1.000000e+00
std,NaN,1.985603e+01,1.559266e+02,7.814341e-01,2.142851,1.96875,1.828814e-02


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

1) Signal 1: Pages ranking in the top 10 average positions (gsc_avg_position <= 10) have a significantly higher average CTR than pages ranking outside the top 10.

2) Signal 2: High GSC impressions (gsc_impressions) strongly correlate with higher organic traffic and pageviews in GA4 (ga4_pageviews).

3) Signal 3: Pages receiving over 1,000 impressions (gsc_impressions >= 1000) will always generate at least 1 click (gsc_clicks > 0).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#Signal 1
threshold = 10
dataset["bucket"] = np.where(
    (dataset["gsc_avg_position"] > 0) & (dataset["gsc_avg_position"] <= threshold),
    "top_10",
    "not_top_10"
)
print(dataset.groupby("bucket")["ctr"].mean())
print("CONFIRMED")

bucket
not_top_10    0.000421
top_10        0.003876
Name: ctr, dtype: float64
CONFIRMED


In [ ]:
#Signal 2
correlation = dataset["gsc_impressions"].corr(dataset["ga4_pageviews"])
print(correlation)
print("FALSE")

0.2891694812760931
FALSE


In [ ]:
#Signal 3
filtered = dataset[dataset["gsc_impressions"] >= 1000]
percentage = (filtered["gsc_clicks"] > 0).mean() * 100
print(f"{percentage:.2f}%")
print("FALSE")

78.44%
FALSE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## 3. The flag-linked test

* **Flag Tested:** "Striking Distance" Rule (`gsc_avg_position` between 4 and 20 AND `gsc_impressions` ≥ 500).
* **Rule Assumption:** Filtering for striking distance positions with high impressions isolates pages with high user intent and active traffic potential compared to background noise.
* **Audit Result:** **CONFIRMED**. Only 4.25% of overall dataset records generate >0 clicks, whereas 75.27% of flagged striking-distance records successfully generate clicks. This rule effectively filters out dead-weight pages and isolates active search opportunities.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

filtered_new = dataset[(dataset["gsc_avg_position"] > 3) & (dataset["gsc_avg_position"] <= 20) & (dataset["gsc_impressions"] >= 500)]
percentage_filtered = (filtered_new["gsc_clicks"] > 0).mean() * 100
percentage_dataset = (dataset["gsc_clicks"] > 0).mean() * 100
print(f"{percentage_filtered:.2f}%")
print(f"{percentage_dataset:.2f}%")


75.27%
4.25%


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

1)Focus on Striking Distance: Content teams should prioritize optimization work on pages flagged in positions 4–20 with high impressions, as over 75% of them already drive active clicks and represent immediate upside.  

2)Don't Assume High Impressions Equal Clicks: Raw impressions alone do not guarantee traffic (as confirmed in Section 2); optimization efforts must specifically target click-through rate fixes (titles/meta descriptions) rather than just pushing visibility.

3)Log-Scale and Filter Dead Weight: Because over 95% of overall records generate zero clicks, future modeling efforts should filter out dead-weight zero-impression rows and apply transformations to handle skewed traffic distributions

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.